In [6]:
from pathlib import Path
import sys
import torch

INPUT = Path("/kaggle/input")
OUT = Path("/kaggle/working/data")
OUT.mkdir(parents=True, exist_ok=True)
IMAGE_EXTS = {".png", ".jpg", ".jpeg"}

mura_matches = list(INPUT.rglob("MURA-v1.1"))
if not mura_matches:
    raise FileNotFoundError("MURA-v1.1 not found under /kaggle/input")
MURA_ROOT = mura_matches[0]

RA_ROOTS = []
for folder in INPUT.rglob("dataset"):
    categories = ("Normal", "Mild", "Moderate", "Severe")
    if (
        all((folder / split).is_dir() for split in ("train", "val", "test"))
        and all((folder / "train" / c).is_dir() for c in categories)
    ):
        RA_ROOTS.append(folder)

if not RA_ROOTS:
    raise FileNotFoundError(
        "Could not find the RA dataset with train/val/test and "
        "Normal/Mild/Moderate/Severe folders."
    )
RA_ROOT = RA_ROOTS[0]

PROJECT_ROOTS = [
    p.parent for p in INPUT.rglob("train.py")
    if (p.parent / "models.py").is_file()
    and (p.parent / "preprocess.py").is_file()
]
if not PROJECT_ROOTS:
    raise FileNotFoundError(
        "Attach the RA-CAD-Pipeline-Scripts dataset containing train.py, "
        "models.py, and preprocess.py."
    )
PROJECT_ROOT = PROJECT_ROOTS[0]
sys.path.insert(0, str(PROJECT_ROOT))

from train import ManifestDataset
from models import DenseNetBinaryClassifier

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("MURA:", MURA_ROOT)
print("RA dataset:", RA_ROOT)
print("Project code:", PROJECT_ROOT)
print("Device:", DEVICE)

MURA: /kaggle/input/datasets/cjinny/mura-v11/MURA-v1.1
RA dataset: /kaggle/input/datasets/sindhuvunna/rheumatoid-arthritis-detection/dataset
Project code: /kaggle/input/datasets/thujainanajumuddin/ra-cad-pipeline-scripts
Device: cuda


In [7]:
import csv

def image_files(folder):
    return sorted(
        p for p in folder.rglob("*")
        if p.is_file()
        and p.suffix.lower() in IMAGE_EXTS
        and not p.name.startswith("._")
    )

def write_manifest(filename, rows):
    path = OUT / filename
    with path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["image_path", "label"])
        writer.writeheader()
        writer.writerows(rows)
    print(f"{path}: {len(rows)} images")
    return path

def mura_normal(split):
    rows = []
    for body_part in ("XR_HAND", "XR_WRIST", "XR_FINGER"):
        body = MURA_ROOT / split / body_part
        if body.exists():
            for folder in body.rglob("study*_negative"):
                rows.extend(
                    {"image_path": str(p), "label": 0}
                    for p in image_files(folder)
                )
    return rows

def ra_split(split):
    rows = []
    for category in ("Normal", "Mild", "Moderate", "Severe"):
        folder = RA_ROOT / split / category
        if folder.exists():
            label = 0 if category == "Normal" else 1
            rows.extend(
                {"image_path": str(p), "label": label}
                for p in image_files(folder)
            )
    return rows

# MURA contributes normal training controls only; its positive studies are excluded.
train_rows = mura_normal("train") + ra_split("train")

# Validation and test come from separate official RA dataset splits.
val_rows = ra_split("val")
test_rows = ra_split("test")

# Kept separate for an additional MURA-control false-positive check.
mura_control_rows = mura_normal("valid")

for name, rows in (
    ("train", train_rows),
    ("validation", val_rows),
    ("test", test_rows),
    ("MURA control check", mura_control_rows),
):
    counts = {label: sum(r["label"] == label for r in rows) for label in (0, 1)}
    print(name, counts)

if not train_rows or not val_rows or not test_rows:
    raise ValueError("A split is empty. Check the dataset paths and folder names.")
if not any(r["label"] == 0 for r in val_rows) or not any(r["label"] == 1 for r in val_rows):
    raise ValueError("RA validation split needs both Normal and RA-labeled images.")
if not any(r["label"] == 0 for r in test_rows) or not any(r["label"] == 1 for r in test_rows):
    raise ValueError("RA test split needs both Normal and RA-labeled images.")

write_manifest("binary_train.csv", train_rows)
write_manifest("binary_val.csv", val_rows)
write_manifest("binary_test.csv", test_rows)
write_manifest("mura_control_check.csv", mura_control_rows)

train {0: 15803, 1: 1996}
validation {0: 608, 1: 426}
test {0: 610, 1: 432}
MURA control check {0: 849, 1: 0}
/kaggle/working/data/binary_train.csv: 17799 images
/kaggle/working/data/binary_val.csv: 1034 images
/kaggle/working/data/binary_test.csv: 1042 images
/kaggle/working/data/mura_control_check.csv: 849 images


PosixPath('/kaggle/working/data/mura_control_check.csv')

In [9]:
from pathlib import Path
import time
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# Training settings
BATCH_SIZE = 16
EPOCHS = 10
WORKERS = 2
LEARNING_RATE = 1e-4

# Prepare datasets and loaders
train_ds = ManifestDataset(OUT / "binary_train.csv", "binary", training=True)
val_ds = ManifestDataset(OUT / "binary_val.csv", "binary", training=False)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
)

# Create the model and optimizer
print(f"Preparing DenseNet121 on {DEVICE}...")
model = DenseNetBinaryClassifier(pretrained=True).to(DEVICE)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4,
)

# Weight the positive class to reduce the effect of class imbalance.
train_labels = np.asarray([labels[0] for _, labels in train_ds.samples])
n_negative = int((train_labels == 0).sum())
n_positive = int((train_labels == 1).sum())

if n_negative == 0 or n_positive == 0:
    raise ValueError(
        f"Training split needs both classes; found normal={n_negative}, RA={n_positive}."
    )

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([n_negative / n_positive], device=DEVICE)
)

# The app can load this checkpoint from its local checkpoints folder.
CHECKPOINT = Path("/kaggle/working/checkpoints/ra_binary.pt")
CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
best_val_loss = float("inf")

print(f"Training images: {len(train_ds):,} | validation images: {len(val_ds):,}")
print(f"Batches per epoch: train={len(train_loader):,}, validation={len(val_loader):,}")
print(f"Checkpoint will be saved to: {CHECKPOINT}")

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()

    # Training
    model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    train_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch}/{EPOCHS} — train",
        unit="batch",
        leave=True,
    )

    for images, labels in train_bar:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels[:, 0].float().to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        batch_size = images.size(0)
        train_loss_sum += loss.item() * batch_size
        train_pred = torch.sigmoid(logits.detach()) >= 0.5
        train_correct += (train_pred == labels.bool()).sum().item()
        train_total += labels.numel()

        train_bar.set_postfix(
            loss=f"{train_loss_sum / max(train_total, 1):.4f}",
            accuracy=f"{train_correct / max(train_total, 1):.3f}",
        )

    train_loss = train_loss_sum / len(train_ds)
    train_accuracy = train_correct / max(train_total, 1)

    # Validation
    model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    val_bar = tqdm(
        val_loader,
        desc=f"Epoch {epoch}/{EPOCHS} — validation",
        unit="batch",
        leave=True,
    )

    with torch.no_grad():
        for images, labels in val_bar:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels[:, 0].float().to(DEVICE, non_blocking=True)

            logits = model(images)
            loss = criterion(logits, labels)

            batch_size = images.size(0)
            val_loss_sum += loss.item() * batch_size
            val_pred = torch.sigmoid(logits) >= 0.5
            val_correct += (val_pred == labels.bool()).sum().item()
            val_total += labels.numel()

            val_bar.set_postfix(
                loss=f"{val_loss_sum / max(val_total, 1):.4f}",
                accuracy=f"{val_correct / max(val_total, 1):.3f}",
            )

    val_loss = val_loss_sum / len(val_ds)
    val_accuracy = val_correct / max(val_total, 1)
    elapsed = time.time() - epoch_start

    print(
        f"Epoch {epoch}/{EPOCHS} complete in {elapsed / 60:.1f} min | "
        f"train loss={train_loss:.4f}, train accuracy={train_accuracy:.3f} | "
        f"val loss={val_loss:.4f}, val accuracy={val_accuracy:.3f}"
    )

    # Keep the model with the lowest validation loss.
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(
            {
                "state_dict": model.state_dict(),
                "epoch": epoch,
                "validation_loss": val_loss,
            },
            CHECKPOINT,
        )
        print(f"Best checkpoint saved: {CHECKPOINT}")

print(f"Training finished. Checkpoint exists: {CHECKPOINT.is_file()}")

Preparing DenseNet121 on cuda...
Training images: 17,799 | validation images: 1,034
Batches per epoch: train=1,113, validation=65
Checkpoint will be saved to: /kaggle/working/checkpoints/ra_binary.pt


Epoch 1/10 — train:   0%|          | 0/1113 [00:00<?, ?batch/s]

Epoch 1/10 — validation:   0%|          | 0/65 [00:00<?, ?batch/s]

Epoch 1/10 complete in 3.7 min | train loss=0.0400, train accuracy=0.991 | val loss=0.0015, val accuracy=1.000
Best checkpoint saved: /kaggle/working/checkpoints/ra_binary.pt


Epoch 2/10 — train:   0%|          | 0/1113 [00:00<?, ?batch/s]

...

Epoch 9/10 — validation:   0%|          | 0/65 [00:00<?, ?batch/s]

Epoch 9/10 complete in 3.9 min | train loss=0.0009, train accuracy=1.000 | val loss=0.0003, val accuracy=1.000


Epoch 10/10 — train:   0%|          | 0/1113 [00:00<?, ?batch/s]

Epoch 10/10 — validation:   0%|          | 0/65 [00:00<?, ?batch/s]

Epoch 10/10 complete in 4.5 min | train loss=0.0002, train accuracy=1.000 | val loss=0.0004, val accuracy=1.000
Training finished. Checkpoint exists: True


In [10]:
from sklearn.metrics import classification_report, confusion_matrix

if not CHECKPOINT.is_file():
    raise FileNotFoundError(
        f"{CHECKPOINT} not found. Wait for Cell 3 to finish and save a checkpoint."
    )

saved = torch.load(CHECKPOINT, map_location=DEVICE, weights_only=False)
model.load_state_dict(saved["state_dict"])
model.eval()

test_ds = ManifestDataset(OUT / "binary_test.csv", "binary", training=False)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
)

# Report unweighted binary cross-entropy as the test loss.
test_criterion = nn.BCEWithLogitsLoss()
test_loss_sum = 0.0
test_correct = 0
test_total = 0
y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Held-out test", unit="batch"):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels[:, 0].float().to(DEVICE, non_blocking=True)

        logits = model(images)
        loss = test_criterion(logits, labels)

        batch_size = images.size(0)
        test_loss_sum += loss.item() * batch_size
        predictions = (torch.sigmoid(logits) >= 0.5).int()

        test_correct += (predictions == labels.int()).sum().item()
        test_total += labels.numel()
        y_true.extend(labels.int().cpu().tolist())
        y_pred.extend(predictions.cpu().tolist())

test_loss = test_loss_sum / len(test_ds)
test_accuracy = test_correct / max(test_total, 1)

print(f"Test loss:     {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.3f}")
print("Confusion matrix (rows=true, columns=predicted; labels Normal, RA):")
print(confusion_matrix(y_true, y_pred, labels=[0, 1]))
print(classification_report(
    y_true, y_pred,
    labels=[0, 1],
    target_names=["Normal", "RA"],
    zero_division=0,
))

print("Checkpoint used:", CHECKPOINT)

Held-out test:   0%|          | 0/66 [00:00<?, ?batch/s]

Test loss:     0.0001
Test accuracy: 1.000
Confusion matrix (rows=true, columns=predicted; labels Normal, RA):
[[610   0]
 [  0 432]]
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00       610
          RA       1.00      1.00      1.00       432

    accuracy                           1.00      1042
   macro avg       1.00      1.00      1.00      1042
weighted avg       1.00      1.00      1.00      1042

Checkpoint used: /kaggle/working/checkpoints/ra_binary.pt


In [15]:
from IPython.display import FileLink
import os

checkpoint_path = "/kaggle/working/checkpoints/ra_binary.pt"

if os.path.exists(checkpoint_path):
    print("File found! Click the link below to download:")
    display(FileLink("checkpoints/ra_binary.pt"))
else:
    print(f"File not found at {checkpoint_path}. Ensure training completed successfully.")

File found! Click the link below to download:


/kaggle/working/checkpoints/ra_binary.pt